# Model 5 - Post Donation Yield (Leakage-Safe)

Goal: predict `donation_referrals` for planned posts.

Leakage guards:
- Uses only features available before/at posting time.
- Excludes post-outcome metrics (`impressions`, `reach`, `likes`, `comments`, `shares`, `saves`, `click_throughs`, `video_views`, `engagement_rate`, etc.).
- Chronological train/test split.


In [7]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PowerTransformer, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
import joblib

DATA_DIR = Path('.')
ARTIFACTS_DIR = DATA_DIR / 'artifacts'
ARTIFACTS_DIR.mkdir(exist_ok=True)

posts = pd.read_csv(DATA_DIR / 'social_media_posts.csv', parse_dates=['created_at'])

def time_split(df, time_col, frac=0.8):
    df = df.sort_values(time_col).copy()
    cut = int(len(df) * frac)
    return df.iloc[:cut].copy(), df.iloc[cut:].copy()

def build_preprocessor(X):
    num_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
    cat_cols = [c for c in X.columns if c not in num_cols]
    return ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('pow', PowerTransformer(method='yeo-johnson', standardize=False)), ('sc', StandardScaler())]), num_cols),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
    ])


In [6]:
feature_cols = [
    'platform', 'day_of_week', 'post_hour', 'post_type', 'media_type', 'num_hashtags',
    'mentions_count', 'has_call_to_action', 'call_to_action_type', 'content_topic',
    'sentiment_tone', 'caption_length', 'features_resident_story', 'campaign_name',
    'is_boosted', 'boost_budget_php'
]
cols = [c for c in feature_cols + ['donation_referrals', 'created_at'] if c in posts.columns]
m5 = posts[cols].copy()

train_df, test_df = time_split(m5, 'created_at', 0.8)
X_train = train_df.drop(columns=['donation_referrals', 'created_at'])
y_train = train_df['donation_referrals']
X_test = test_df.drop(columns=['donation_referrals', 'created_at'])
y_test = test_df['donation_referrals']

pre = build_preprocessor(X_train)
predictive = Pipeline([('pre', pre), ('model', RandomForestRegressor(n_estimators=350, min_samples_leaf=4, random_state=42))])
explanatory = Pipeline([('pre', pre), ('model', LinearRegression())])

predictive.fit(X_train, y_train)
explanatory.fit(X_train, y_train)

pred = predictive.predict(X_test)
mse = mean_squared_error(y_test, pred)
rmse = mse ** 0.5
print({'mae': mean_absolute_error(y_test, pred), 'rmse': rmse, 'r2': r2_score(y_test, pred)})

imp = permutation_importance(predictive, X_test, y_test, n_repeats=8, random_state=42)
print(pd.DataFrame({'feature': X_test.columns, 'importance': imp.importances_mean}).sort_values('importance', ascending=False).head(10))

joblib.dump(predictive, ARTIFACTS_DIR / 'model5_predictive.joblib')
joblib.dump(explanatory, ARTIFACTS_DIR / 'model5_explanatory.joblib')


{'mae': 10.964377190914284, 'rmse': 24.652365777660414, 'r2': 0.3446629542275066}
                    feature  importance
12  features_resident_story    0.514682
2                 post_hour    0.113457
3                 post_type    0.061437
0                  platform    0.025907
10           sentiment_tone    0.016299
15         boost_budget_php    0.008253
8       call_to_action_type    0.006527
5              num_hashtags    0.004852
4                media_type    0.004339
7        has_call_to_action    0.003329


['artifacts\\model5_explanatory.joblib']

In [ ]:
# Final business insights block (human-readable + actionable)

print('\n=== BUSINESS TAKEAWAYS: MODEL 5 (POST DONATION YIELD) ===')

# Build prediction table on all posts for strategy insights
scored = m5.copy()
feature_inputs = [c for c in scored.columns if c not in ['donation_referrals', 'created_at']]
scored['pred_referrals'] = predictive.predict(scored[feature_inputs])

baseline_pred = float(scored['pred_referrals'].mean())
print(f'Baseline expected referrals/post (model): {baseline_pred:.2f}')

# 1) Best 5 posting windows (day + hour)
win_tbl = (
    scored.groupby(['day_of_week', 'post_hour'], dropna=False)['pred_referrals']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
    .head(5)
)
win_tbl['expected_uplift_vs_baseline_pct'] = ((win_tbl['pred_referrals'] / baseline_pred) - 1.0) * 100
print('\nBest 5 posting windows:')
display(win_tbl)

# 2) Best post types by platform
ptype_tbl = (
    scored.groupby(['platform', 'post_type'], dropna=False)['pred_referrals']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
best_ptype_per_platform = ptype_tbl.groupby('platform', as_index=False).head(1).copy()
best_ptype_per_platform['expected_uplift_vs_baseline_pct'] = ((best_ptype_per_platform['pred_referrals'] / baseline_pred) - 1.0) * 100
print('\nBest post type by platform:')
display(best_ptype_per_platform.sort_values('pred_referrals', ascending=False))

# 3) Story vs no-story uplift
if 'features_resident_story' in scored.columns:
    story_tbl = scored.groupby('features_resident_story', dropna=False)['pred_referrals'].mean().reset_index()
    print('\nResident story effect (predicted):')
    display(story_tbl)

# 4) Executive-readable guidance
top_window = win_tbl.iloc[0]
print('\nActionable guidance:')
print(f"- Prioritize posting around {top_window['day_of_week']} at hour {int(top_window['post_hour']) if pd.notna(top_window['post_hour']) else 'N/A'}.")
print('- Use the best post type per platform table to choose format before publishing.')
print('- Compare expected uplift vs baseline to prioritize high-leverage content in weekly planning.')
print('- Treat these as predictive recommendations (ranking), not causal proof.')